# LeetCode 1602 — Find Nearest Right Node in Binary Tree bằng Python

Link đề: https://leetcode.com/problems/find-nearest-right-node-in-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Breadth-First Search, Binary Tree

> **Ghi chú Premium:** đây là bài Premium. Đề bài và chữ ký hàm được viết theo hiểu biết về bài (lấy từ trang doocs: https://leetcode.doocs.org/en/lc/1602/), nên đối chiếu lại nếu bạn có tài khoản Premium. Tham số `u` là **một đối tượng `TreeNode`** trong cây (không phải giá trị).

Skeleton LeetCode (theo lời giải Python của doocs):

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findNearestRightNode(self, root: TreeNode, u: TreeNode) -> Optional[TreeNode]:
        ...
```

## 1. Mục tiêu học

1. Dựng cây nhị phân từ mảng level-order (có `None`) và tìm **đối tượng nút** theo giá trị để làm tham số `u`.
2. Nắm **BFS theo tầng** với `deque`: xử lý đúng `len(q)` nút mỗi tầng và biết khi nào một tầng **kết thúc**.
3. Áp dụng BFS để trả về **nút kế bên phải cùng tầng** của `u`, hoặc `None` nếu `u` là nút phải nhất của tầng.
4. Giải bằng **DFS tiền thứ tự (trái trước)**: ghi độ sâu của `u`, nút đầu tiên gặp sau đó ở cùng độ sâu là đáp án.
5. Phân biệt so sánh **đối tượng** (`is`) và **giá trị**; tránh các lỗi "tràn sang tầng sau".

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây nhị phân và một nút `u` thuộc cây. Trả về **nút gần nhất cùng tầng nằm bên phải** `u`, hoặc trả `null` (`None`) nếu `u` là nút **phải nhất** trong tầng của nó.

**Ví dụ 1:** `root = [1,2,3,null,4,5,6]`, `u = 4` → `5`. Tầng thứ 3 gồm `4, 5, 6`; ngay bên phải `4` là `5`.

**Ví dụ 2:** `root = [3,null,4,2]`, `u = 2` → `null`. Không có nút nào bên phải `2`.

**Ràng buộc:**
- Số nút trong khoảng `[1, 10⁵]`.
- `1 ≤ Node.val ≤ 10⁵`.
- Mọi giá trị trong cây **đôi một khác nhau**.
- `u` là một nút của cây có gốc `root`.

In [1]:
import random
import time
from collections import deque
from typing import Optional, List

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0]); q = deque([root]); i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

def find_node(root, val):
    """Tìm ĐỐI TƯỢNG nút có giá trị val (dùng để tạo tham số u)."""
    stack = [root]
    while stack:
        x = stack.pop()
        if x is None: continue
        if x.val == val: return x
        stack += [x.left, x.right]
    return None

examples = [([1, 2, 3, None, 4, 5, 6], 4, 5), ([3, None, 4, 2], 2, None)]
for arr, uval, want in examples:
    root = build_tree(arr); u = find_node(root, uval)
    print(f"root={arr}, u={u} -> mong đợi {want}")

root=[1, 2, 3, None, 4, 5, 6], u=TreeNode(4) -> mong đợi 5
root=[3, None, 4, 2], u=TreeNode(2) -> mong đợi None


## 3. Kiến thức nền

### 3.1 `deque` — hàng đợi hai đầu

`append` (thêm cuối) và `popleft` (lấy đầu) đều O(1). Dùng `list.pop(0)` thì O(n) — chậm với `10⁵` nút. `q[0]` xem phần tử đầu mà không lấy ra.

In [2]:
q = deque([1, 2])
q.append(3)
print(q.popleft(), q, "đầu hàng:", q[0])

1 deque([2, 3]) đầu hàng: 2


### 3.2 BFS theo tầng

Ở đầu mỗi tầng, hàng đợi chứa **đúng** các nút của tầng đó. Ghi lại `size = len(q)`, rồi lấy ra đúng `size` nút; con của chúng được đẩy vào cuối (thuộc tầng sau).

In [3]:
def levels(root):
    res, q = [], deque([root])
    while q:
        size = len(q)
        cur = []
        for _ in range(size):
            x = q.popleft(); cur.append(x.val)
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        res.append(cur)
    return res

print(levels(build_tree([1, 2, 3, None, 4, 5, 6])))
print(levels(build_tree([3, None, 4, 2])))

[[1], [2, 3], [4, 5, 6]]
[[3], [4], [2]]


### 3.3 So sánh đối tượng: `is` và `==`

`u` là **một đối tượng**. `x is u` kiểm tra hai biến cùng trỏ **một nút**. Lớp `TreeNode` không định nghĩa `__eq__` nên `==` cũng so sánh danh tính — nhưng `is` nói rõ ý định hơn. Hai nút khác nhau có cùng giá trị thì `is` là `False`.

In [4]:
a, b = TreeNode(7), TreeNode(7)
print(a is a, a is b, a == b, a.val == b.val)

True False False True


### 3.4 DFS bằng stack và độ sâu

Lưu cặp `(nút, độ sâu)` trong stack. Muốn thăm **trái trước** (tiền thứ tự trái → phải) thì đẩy **phải trước, trái sau**.

In [5]:
def preorder_with_depth(root):
    out, st = [], [(root, 0)]
    while st:
        x, d = st.pop()
        out.append((x.val, d))
        if x.right: st.append((x.right, d + 1))
        if x.left: st.append((x.left, d + 1))
    return out

print(preorder_with_depth(build_tree([1, 2, 3, None, 4, 5, 6])))

[(1, 0), (2, 1), (4, 2), (3, 1), (5, 2), (6, 2)]


## 4. Ý tưởng nền tảng / trực giác

Vẽ ví dụ 1:

```
        1          tầng 0
      /   \
     2     3       tầng 1
      \   / \
       4 5   6     tầng 2
```

Các tầng đọc từ trái sang phải: `[1]`, `[2, 3]`, `[4, 5, 6]`. `u = 4` ở tầng 2, vị trí 0 ⇒ nút bên phải là `5`.

Ví dụ 2: `[3]`, `[4]`, `[2]` — `2` là nút duy nhất (cũng là phải nhất) của tầng ⇒ `None`.

**Chú ý:** "bên phải gần nhất" là theo **thứ tự trong tầng**, không cần là anh em ruột. Ở ví dụ 1, `4` là con phải của `2`, còn `5` là con trái của `3` — khác cha nhưng vẫn là hàng xóm trong tầng.

In [6]:
root = build_tree([1, 2, 3, None, 4, 5, 6])
for d, lv in enumerate(levels(root)):
    pairs = [f"{lv[i]}→{lv[i+1] if i + 1 < len(lv) else None}" for i in range(len(lv))]
    print(f"tầng {d}: {lv}   hàng xóm phải: {pairs}")

tầng 0: [1]   hàng xóm phải: ['1→None']
tầng 1: [2, 3]   hàng xóm phải: ['2→3', '3→None']
tầng 2: [4, 5, 6]   hàng xóm phải: ['4→5', '5→6', '6→None']


## 5. Cách 1 — Brute Force / baseline: gom toàn bộ các tầng rồi tra

**Ý tưởng:** BFS lưu **mọi tầng** dưới dạng list các nút. Tìm tầng chứa `u`, tìm vị trí `u` trong tầng, trả nút kế tiếp (hoặc `None`).

Cách này đúng và dễ hiểu, nhưng tốn bộ nhớ lưu cả cây và luôn duyệt hết cây kể cả khi `u` ở gần gốc.

In [7]:
def nearest_right_levels(root, u):
    all_levels, q = [], deque([root])
    while q:
        size = len(q); cur = []
        for _ in range(size):
            x = q.popleft(); cur.append(x)
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        all_levels.append(cur)
    for lv in all_levels:
        for i, x in enumerate(lv):
            if x is u:
                return lv[i + 1] if i + 1 < len(lv) else None
    return None

for arr, uval, want in examples:
    root = build_tree(arr)
    print(nearest_right_levels(root, find_node(root, uval)), "| mong đợi", want)

TreeNode(5) | mong đợi 5
None | mong đợi None


**Độ phức tạp:** Time O(n) (duyệt toàn bộ cây, rồi tìm), Space O(n) (lưu mọi nút trong `all_levels`).

**Nút thắt:** lưu tất cả là thừa — khi gặp `u` trong BFS, hàng xóm phải của nó **đang đứng ngay đầu hàng đợi** (nếu tầng chưa hết). Có thể dừng ngay.

Một baseline "ngây thơ" hơn nữa: với mỗi nút, tính độ sâu bằng cách tìm từ gốc (O(n) mỗi nút) ⇒ O(n²). Ta không đi theo hướng đó.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (BFS):** trong BFS theo tầng, khi lấy ra nút thứ `k` của tầng (tầng có `size` nút), hàng đợi bắt đầu bằng **các nút còn lại của tầng này** (nút `k+1`, …), sau đó mới đến con cháu (tầng sau). Vậy khi lấy ra `u`:
- **Nếu** `u` chưa phải nút cuối của tầng (còn `size − k − 1 > 0` nút) **thì** đáp án là `q[0]`.
- **Ngược lại** thì đáp án là `None` — dù `q` có thể không rỗng (chứa tầng sau).

Lời giải doocs đếm ngược `i` từ `len(q) − 1` về 0: `i` = số nút **còn lại** của tầng sau nút vừa lấy ⇒ `return q[0] if i else None`.

**Insight (DFS):** duyệt tiền thứ tự **trái trước**: trong cùng một tầng, các nút được gặp theo thứ tự từ trái sang phải. Ghi `depth_u` khi gặp `u`; **nút đầu tiên** gặp sau đó có độ sâu bằng `depth_u` chính là hàng xóm phải. Nếu không gặp ⇒ `None`.

**Invariant BFS:** ở đầu mỗi vòng `for` của một tầng, `q` = [phần còn lại của tầng hiện tại] + [con của các nút đã lấy ra ở tầng này].

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS theo tầng, dừng sớm (lời giải doocs)

Từng đoạn:
1. `q = deque([root])`.
2. Mỗi tầng: `for i in range(len(q) - 1, -1, -1)` — `i` là số nút còn lại sau nút đang lấy.
3. Lấy `x = q.popleft()`; nếu `x is u` ⇒ `return q[0] if i else None`.
4. Đẩy con trái, con phải.

In [8]:
def nearest_right_bfs(root, u):
    q = deque([root])
    while q:
        for i in range(len(q) - 1, -1, -1):
            x = q.popleft()
            if x is u:
                return q[0] if i else None
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
    return None

for arr, uval, want in examples:
    root = build_tree(arr)
    print(nearest_right_bfs(root, find_node(root, uval)), "| mong đợi", want)

TreeNode(5) | mong đợi 5
None | mong đợi None


Code nộp LeetCode của Cách 2 (xem mục 9).

**Độ phức tạp:** Time O(n) worst (dừng ngay khi gặp `u`), Space O(w) với `w` là độ rộng lớn nhất của cây (≤ n).

### 7.2 Cách 3 — DFS tiền thứ tự (trái trước) ghi độ sâu của `u`

Từng đoạn:
1. Stack chứa `(nút, độ sâu)`; đẩy phải trước, trái sau ⇒ lấy ra theo thứ tự trái trước.
2. `depth_u = None`. Khi lấy ra một nút:
   - nếu `depth_u` đã biết và độ sâu nút bằng `depth_u` ⇒ trả nút đó (gặp đầu tiên = gần nhất bên phải);
   - nếu nút là `u` ⇒ ghi `depth_u`.
3. Hết stack ⇒ `None`.

Dùng stack tường minh để không bị giới hạn đệ quy (cây lệch có thể sâu `10⁵`).

In [9]:
def nearest_right_dfs(root, u):
    depth_u = None
    st = [(root, 0)]
    while st:
        x, d = st.pop()
        if depth_u is not None and d == depth_u:
            return x
        if x is u:
            depth_u = d
        if x.right: st.append((x.right, d + 1))
        if x.left: st.append((x.left, d + 1))
    return None

for arr, uval, want in examples:
    root = build_tree(arr)
    print(nearest_right_dfs(root, find_node(root, uval)), "| mong đợi", want)

TreeNode(5) | mong đợi 5
None | mong đợi None


Vì sao "nút đầu tiên cùng độ sâu sau `u`" là hàng xóm phải? Trong tiền thứ tự trái-trước, mọi nút ở tầng `d` xuất hiện theo đúng thứ tự trái → phải (nút bên trái nằm trong cây con được thăm trước). Nên nút tầng `d` kế tiếp sau `u` trong thứ tự duyệt chính là nút kế tiếp trong tầng.

Code nộp LeetCode của Cách 3:

```python
class Solution:
    def findNearestRightNode(self, root: TreeNode, u: TreeNode) -> Optional[TreeNode]:
        depth_u = None
        st = [(root, 0)]
        while st:
            x, d = st.pop()
            if depth_u is not None and d == depth_u:
                return x
            if x is u:
                depth_u = d
            if x.right:
                st.append((x.right, d + 1))
            if x.left:
                st.append((x.left, d + 1))
        return None
```

**Độ phức tạp:** Time O(n), Space O(h) (h = chiều cao) cho stack — lợi hơn BFS khi cây rộng.

### 7.3 Cách 4 — BFS không đếm tầng: hàng đợi chứa `(nút, độ sâu)`

Thay vì đếm `size`, mỗi phần tử mang theo độ sâu. Khi lấy ra `u` với độ sâu `d`: nếu `q` khác rỗng **và** phần tử đầu có cùng độ sâu `d` ⇒ đó là đáp án; ngược lại `None`. Cách này dễ đọc và khó sai "ranh giới tầng".

In [10]:
def nearest_right_bfs_depth(root, u):
    q = deque([(root, 0)])
    while q:
        x, d = q.popleft()
        if x is u:
            return q[0][0] if q and q[0][1] == d else None
        if x.left: q.append((x.left, d + 1))
        if x.right: q.append((x.right, d + 1))
    return None

for arr, uval, want in examples:
    root = build_tree(arr)
    print(nearest_right_bfs_depth(root, find_node(root, uval)), "| mong đợi", want)

TreeNode(5) | mong đợi 5
None | mong đợi None


Code nộp LeetCode của Cách 4:

```python
class Solution:
    def findNearestRightNode(self, root: TreeNode, u: TreeNode) -> Optional[TreeNode]:
        q = deque([(root, 0)])
        while q:
            x, d = q.popleft()
            if x is u:
                return q[0][0] if q and q[0][1] == d else None
            if x.left:
                q.append((x.left, d + 1))
            if x.right:
                q.append((x.right, d + 1))
        return None
```

**Độ phức tạp:** Time O(n), Space O(w).

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1 (`u = 4`):

| tầng | i | lấy ra | là u? | q sau khi lấy (trước khi đẩy con) | đẩy | q sau |
|---|---|---|---|---|---|---|
| 0 | 0 | 1 | không | [] | 2, 3 | [2, 3] |
| 1 | 1 | 2 | không | [3] | 4 | [3, 4] |
| 1 | 0 | 3 | không | [4] | 5, 6 | [4, 5, 6] |
| 2 | 2 | **4** | **có** | [5, 6] | — | i = 2 ≠ 0 ⇒ trả `q[0] = 5` |

In [11]:
def trace_bfs(root, u):
    q = deque([root]); level = 0
    while q:
        print(f"-- tầng {level}: q = {[x.val for x in q]}")
        for i in range(len(q) - 1, -1, -1):
            x = q.popleft()
            if x is u:
                ans = q[0] if i else None
                print(f"   lấy {x.val} (i={i}) = u -> còn {i} nút cùng tầng -> trả {ans}")
                return ans
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
            print(f"   lấy {x.val} (i={i}), q = {[y.val for y in q]}")
        level += 1
    return None

root = build_tree([1, 2, 3, None, 4, 5, 6])
trace_bfs(root, find_node(root, 4))

-- tầng 0: q = [1]
   lấy 1 (i=0), q = [2, 3]
-- tầng 1: q = [2, 3]
   lấy 2 (i=1), q = [3, 4]
   lấy 3 (i=0), q = [4, 5, 6]
-- tầng 2: q = [4, 5, 6]
   lấy 4 (i=2) = u -> còn 2 nút cùng tầng -> trả TreeNode(5)


TreeNode(5)

In [12]:
root = build_tree([3, None, 4, 2])
trace_bfs(root, find_node(root, 2))
print()
root = build_tree([3, None, 4, 2])
trace_bfs(root, find_node(root, 4))       # u = 4 là phải nhất tầng 1 -> None (con 2 của nó chưa được đẩy vào hàng đợi)

-- tầng 0: q = [3]
   lấy 3 (i=0), q = [4]
-- tầng 1: q = [4]
   lấy 4 (i=0), q = [2]
-- tầng 2: q = [2]
   lấy 2 (i=0) = u -> còn 0 nút cùng tầng -> trả None

-- tầng 0: q = [3]
   lấy 3 (i=0), q = [4]
-- tầng 1: q = [4]
   lấy 4 (i=0) = u -> còn 0 nút cùng tầng -> trả None


In [13]:
def trace_dfs(root, u):
    depth_u = None; st = [(root, 0)]
    while st:
        x, d = st.pop()
        print(f"thăm {x.val} (độ sâu {d}), depth_u={depth_u}")
        if depth_u is not None and d == depth_u:
            print(f"=> nút đầu tiên cùng độ sâu sau u: {x.val}"); return x
        if x is u:
            depth_u = d; print(f"   gặp u, ghi depth_u = {d}")
        if x.right: st.append((x.right, d + 1))
        if x.left: st.append((x.left, d + 1))
    print("=> không có, trả None"); return None

root = build_tree([1, 2, 3, None, 4, 5, 6])
trace_dfs(root, find_node(root, 4))

thăm 1 (độ sâu 0), depth_u=None
thăm 2 (độ sâu 1), depth_u=None
thăm 4 (độ sâu 2), depth_u=None
   gặp u, ghi depth_u = 2
thăm 3 (độ sâu 1), depth_u=2
thăm 5 (độ sâu 2), depth_u=2
=> nút đầu tiên cùng độ sâu sau u: 5


TreeNode(5)

## 9. Code nộp LeetCode

```python
from collections import deque
from typing import Optional

# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findNearestRightNode(self, root: TreeNode, u: TreeNode) -> Optional[TreeNode]:
        q = deque([root])
        while q:
            for i in range(len(q) - 1, -1, -1):   # i = số nút còn lại của tầng
                x = q.popleft()
                if x is u:
                    return q[0] if i else None
                if x.left:
                    q.append(x.left)
                if x.right:
                    q.append(x.right)
        return None
```

In [14]:
class Solution:
    def findNearestRightNode(self, root: TreeNode, u: TreeNode) -> Optional[TreeNode]:
        q = deque([root])
        while q:
            for i in range(len(q) - 1, -1, -1):
                x = q.popleft()
                if x is u:
                    return q[0] if i else None
                if x.left:
                    q.append(x.left)
                if x.right:
                    q.append(x.right)
        return None

def val_or_none(node):
    return None if node is None else node.val

for arr, uval, want in examples:
    root = build_tree(arr)
    got = val_or_none(Solution().findNearestRightNode(root, find_node(root, uval)))
    print(arr, "u =", uval, "->", got, "OK" if got == want else "SAI")

[1, 2, 3, None, 4, 5, 6] u = 4 -> 5 OK
[3, None, 4, 2] u = 2 -> None OK


## 10. Vì sao thuật toán đúng

**BFS (Cách 2):** chứng minh invariant bằng quy nạp theo tầng: đầu tầng `d`, `q` chứa đúng các nút tầng `d` theo thứ tự trái → phải (tầng 0: chỉ gốc; nếu đúng cho tầng `d` thì khi lấy lần lượt các nút tầng `d` và đẩy con trái rồi con phải, các nút tầng `d+1` được đẩy theo đúng thứ tự trái → phải). Trong lúc xử lý tầng `d`, sau khi lấy nút có `i` nút còn lại, `q` = [`i` nút còn lại của tầng `d`] + [các nút tầng `d+1` đã đẩy]. Nên khi lấy ra `u`: nếu `i > 0` thì `q[0]` là nút tầng `d` ngay bên phải `u`; nếu `i = 0` thì `u` là nút cuối của tầng ⇒ `None`.

**DFS (Cách 3):** tiền thứ tự trái-trước liệt kê các nút của mỗi tầng theo thứ tự trái → phải (với hai nút cùng tầng `a` trái `b`, tổ tiên chung gần nhất có `a` ở cây con trái, `b` ở cây con phải; cây con trái được thăm xong trước). Do đó nút cùng tầng đầu tiên gặp sau `u` là hàng xóm phải gần nhất; không gặp ⇒ `u` phải nhất.

## 11. Phân tích Time / Space

Gọi `n` = số nút, `h` = chiều cao, `w` = độ rộng lớn nhất.

| Cách | Time | Space |
|---|---|---|
| 1. Gom mọi tầng | O(n) luôn duyệt hết | O(n) |
| 2. BFS dừng sớm | O(n) worst | O(w) |
| 3. DFS ghi độ sâu | O(n) worst | O(h) |
| 4. BFS (nút, độ sâu) | O(n) worst | O(w) |

Cây đầy đủ: `w ≈ n/2`, `h ≈ log n` ⇒ DFS tiết kiệm bộ nhớ hơn. Cây lệch (chuỗi): `w = 1`, `h = n` ⇒ BFS tiết kiệm hơn. Đo với `10⁵` nút:

In [15]:
def random_tree(n, rnd):
    """Cây ngẫu nhiên n nút, giá trị 1..n đôi một khác nhau."""
    vals = list(range(1, n + 1)); rnd.shuffle(vals)
    nodes = [TreeNode(v) for v in vals]
    free = [(nodes[0], 0), (nodes[0], 1)]       # các "khe" con trống
    for x in nodes[1:]:
        k = rnd.randrange(len(free))
        parent, side = free[k]
        free[k] = free[-1]; free.pop()
        if side == 0: parent.left = x
        else: parent.right = x
        free += [(x, 0), (x, 1)]
    return nodes[0], nodes

rnd = random.Random(0)
big_root, big_nodes = random_tree(10**5, rnd)
u = big_nodes[-1]
for f in [nearest_right_levels, nearest_right_bfs, nearest_right_dfs, nearest_right_bfs_depth]:
    t0 = time.perf_counter(); r = f(big_root, u)
    print(f"{f.__name__:24s} -> {val_or_none(r)}, {time.perf_counter() - t0:.3f}s")

nearest_right_levels     -> 43920, 0.016s
nearest_right_bfs        -> 43920, 0.011s
nearest_right_dfs        -> 43920, 0.003s
nearest_right_bfs_depth  -> 43920, 0.013s


## 12. Bộ test

Ví dụ đề, edge case (cây 1 nút, `u` là gốc, `u` là nút phải nhất có tầng sau không rỗng, hàng xóm khác cha, cây lệch sâu), và đối chiếu ngẫu nhiên **mọi nút `u`** trên nhiều cây ngẫu nhiên với Cách 1 làm chuẩn.

In [16]:
ALL = [nearest_right_levels, nearest_right_bfs, nearest_right_dfs, nearest_right_bfs_depth,
       lambda r, u: Solution().findNearestRightNode(r, u)]

fixed = examples + [
    ([1], 1, None),                              # một nút
    ([1, 2, 3], 1, None),                        # u là gốc
    ([1, 2, 3], 2, 3),
    ([1, 2, 3, 4, None, None, 5], 4, 5),         # khác cha, cách xa
    ([1, 2, 3, 4], 3, None),                     # phải nhất, tầng sau không rỗng
    ([1, 2, 3, None, None, 4, 5], 2, 3),
    ([1, 2, None, 3, None, 4], 4, None),         # chuỗi lệch trái
]
for arr, uval, want in fixed:
    for f in ALL:
        root = build_tree(arr)
        assert val_or_none(f(root, find_node(root, uval))) == want, (f, arr, uval)
print("Ca cố định:", len(fixed), "OK")

rnd = random.Random(2024)
cnt = 0
for _ in range(300):
    n = rnd.randint(1, 30)
    root, nodes = random_tree(n, rnd)
    for u in nodes:
        want = nearest_right_levels(root, u)
        for f in ALL[1:]:
            assert f(root, u) is want, (f, u)          # so sánh ĐỐI TƯỢNG trả về
        cnt += 1
print("Đã đối chiếu", cnt, "cặp (cây, u) ngẫu nhiên")

Ca cố định: 9 OK
Đã đối chiếu 4486 cặp (cây, u) ngẫu nhiên


In [17]:
# Cây lệch sâu 1e5 (chuỗi phải): cách dùng stack/deque không bị giới hạn đệ quy
chain = TreeNode(1); cur = chain
for v in range(2, 100001):
    cur.right = TreeNode(v); cur = cur.right
for f in ALL[1:]:
    assert f(chain, cur) is None
# Cây đầy đủ 2^16 - 1 nút: nút trái nhất tầng cuối có hàng xóm phải
full = build_tree(list(range(1, 2**16)))
leftmost = full
while leftmost.left: leftmost = leftmost.left
for f in ALL:
    assert val_or_none(f(full, leftmost)) == leftmost.val + 1
print("Cây lệch sâu và cây đầy đủ lớn: OK")

Cây lệch sâu và cây đầy đủ lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả `q[0]` mà không kiểm tra tầng đã hết chưa

In [18]:
def wrong_no_level_check(root, u):
    q = deque([root])
    while q:
        x = q.popleft()
        if x is u:
            return q[0] if q else None            # SAI: q[0] có thể thuộc tầng sau
        if x.left: q.append(x.left)
        if x.right: q.append(x.right)
    return None

root = build_tree([1, 2, 3, 4])
u = find_node(root, 3)
print("sai :", val_or_none(wrong_no_level_check(root, u)), "| đúng:", val_or_none(Solution().findNearestRightNode(root, u)))

sai : 4 | đúng: None


**Vì sao sai:** với `[1,2,3,4]`, `u = 3` là nút phải nhất tầng 1; lúc lấy `3` ra, `q = [4]` (con của `2`, tầng 2). Code sai trả `4` — một nút **ở tầng dưới**. **Sửa:** phải biết ranh giới tầng (đếm `i` như Cách 2 hoặc so độ sâu như Cách 4).

### Lỗi 2 — DFS thăm con phải trước

In [19]:
def wrong_dfs_right_first(root, u):
    depth_u = None; st = [(root, 0)]
    while st:
        x, d = st.pop()
        if depth_u is not None and d == depth_u:
            return x
        if x is u:
            depth_u = d
        if x.left: st.append((x.left, d + 1))     # SAI: đẩy trái trước -> lấy phải trước
        if x.right: st.append((x.right, d + 1))
    return None

root = build_tree([1, 2, 3, None, 4, 5, 6])
u = find_node(root, 5)
print("sai :", val_or_none(wrong_dfs_right_first(root, u)), "| đúng:", val_or_none(Solution().findNearestRightNode(root, u)))

sai : 4 | đúng: 6


**Vì sao sai:** duyệt phải-trước liệt kê mỗi tầng từ **phải sang trái** ⇒ "nút đầu tiên sau `u`" là hàng xóm **trái** (`4` thay vì `6`). **Sửa:** với stack, đẩy **phải trước, trái sau**.

### Lỗi 3 — Duyệt hết DFS và lấy nút cuối cùng cùng độ sâu

In [20]:
def wrong_dfs_last(root, u):
    depth_u, found, st = None, None, [(root, 0)]
    order = []
    while st:
        x, d = st.pop(); order.append((x, d))
        if x is u: depth_u = d
        if x.right: st.append((x.right, d + 1))
        if x.left: st.append((x.left, d + 1))
    seen_u = False
    for x, d in order:
        if x is u: seen_u = True
        elif seen_u and d == depth_u:
            found = x                              # SAI: ghi đè -> lấy nút XA nhất
    return found

root = build_tree([1, 2, 3, None, 4, 5, 6])
u = find_node(root, 4)
print("sai :", val_or_none(wrong_dfs_last(root, u)), "| đúng:", val_or_none(Solution().findNearestRightNode(root, u)))

sai : 6 | đúng: 5


**Vì sao sai:** đề hỏi nút **gần nhất** bên phải ⇒ phải dừng ở nút **đầu tiên** cùng tầng sau `u`; ghi đè liên tục cho nút **phải nhất** (`6`). **Sửa:** `return` ngay lần đầu gặp.

### Lỗi 4 — So sánh theo giá trị với `u` truyền sai kiểu / trả về giá trị thay vì nút

In [21]:
def wrong_return_value(root, u):
    q = deque([root])
    while q:
        for i in range(len(q) - 1, -1, -1):
            x = q.popleft()
            if x.val == u:                        # SAI: coi u là số nguyên
                return q[0].val if i else None    # SAI: trả số thay vì TreeNode
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
    return None

root = build_tree([1, 2, 3, None, 4, 5, 6])
u = find_node(root, 4)
r = wrong_return_value(root, u)
print("sai : kết quả =", r, "(không tìm thấy vì so TreeNode với int)")
print("đúng:", Solution().findNearestRightNode(root, u))

sai : kết quả = None (không tìm thấy vì so TreeNode với int)
đúng: TreeNode(5)


**Vì sao sai:** theo chữ ký (doocs), `u` là một **`TreeNode`**, và hàm phải trả về **`TreeNode`** (hoặc `None`). So `x.val == u` (int với TreeNode) luôn `False` ⇒ không bao giờ tìm thấy; và trả `.val` sai kiểu. **Sửa:** so `x is u`, trả chính nút `q[0]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"cùng tầng" / "level"** trong cây ⇒ BFS theo tầng (đếm `len(q)` đầu mỗi tầng) hoặc DFS mang theo độ sâu.
- **"gần nhất bên phải cùng tầng"** ⇒ khi lấy `u` ra khỏi hàng đợi, câu trả lời đang nằm ở đầu hàng — chỉ cần biết tầng còn nút hay không.
- DFS trái-trước liệt kê mỗi tầng trái → phải: hữu ích cho "right side view" (lấy nút **cuối**), "left side view" (nút **đầu**), "hàng xóm phải".
- Với cây tới `10⁵` nút trong Python: tránh đệ quy sâu, dùng `deque`/stack.
- Đọc kỹ kiểu tham số: `u` là **nút**, không phải giá trị.

## 15. Biến thể của bài

1. **LC 116 / 117 — Populating Next Right Pointers in Each Node (I / II):** gán `next` cho **mọi** nút = hàng xóm phải (bài này là truy vấn một nút).
2. **LC 199 — Binary Tree Right Side View:** nút phải nhất mỗi tầng.
3. **LC 102 — Binary Tree Level Order Traversal.**
4. **Hàng xóm trái gần nhất** (bài tập 3).
5. **Hàng xóm phải thứ `k`** cùng tầng (bài tập 5).
6. **Truy vấn nhiều `u`**: tiền xử lý một lần bằng BFS tạo `dict` nút → hàng xóm phải (bài tập 4).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gom mọi tầng | BFS lưu hết, rồi tìm | O(n) | O(n) | baseline, dễ kiểm chứng |
| 2. BFS dừng sớm | `i` = số nút còn lại của tầng | O(n) | O(w) | **nộp bài** (giống doocs) |
| 3. DFS trái-trước | nút đầu tiên cùng độ sâu sau `u` | O(n) | O(h) | cây rộng, muốn tiết kiệm bộ nhớ |
| 4. BFS (nút, độ sâu) | so độ sâu phần tử đầu hàng | O(n) | O(w) | dễ đọc, khó sai ranh giới tầng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Với `root = [1,2,3,4,5,6,7]`: hàng xóm phải của `5`? của `3`? của `7`? của `1`?

**Bài 2 (tính tay):** Viết trạng thái hàng đợi sau mỗi lần `popleft` khi chạy Cách 2 với `root = [1,2,3,4,null,null,5]`, `u = 4`.

**Bài 3 (điền TODO):** Hoàn thiện `nearest_left(root, u)` trả về hàng xóm **trái** gần nhất cùng tầng.

**Bài 4 (tự viết lại):** Viết `right_neighbors(root)` trả `dict` ánh xạ **giá trị** mỗi nút → giá trị hàng xóm phải (hoặc `None`), bằng một lần BFS.

**Bài 5 (mini challenge):** Viết `kth_right(root, u, k)` trả nút thứ `k` bên phải `u` trong cùng tầng (`k = 1` là bài gốc), hoặc `None`.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def nearest_left(root, u):
    q = deque([root])
    while q:
        prev = None                     # nút vừa lấy trước đó trong CÙNG tầng
        for _ in range(len(q)):
            x = q.popleft()
            # TODO: nếu x is u thì trả prev
            # TODO: cập nhật prev = x, đẩy con trái/phải
            pass
    return None

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def right_neighbors(root):
    # TODO
    pass

def kth_right(root, u, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: liệt kê các tầng. Mức 2: `[1]`, `[2,3]`, `[4,5,6,7]`. Mức 3: `5→6`, `3→None`, `7→None`, `1→None`.

**Bài 2.** Mức 1: bắt đầu `q=[1]`. Mức 2: lấy 1 → `[2,3]`; lấy 2 → `[3,4]`; lấy 3 → `[4,5]`. Mức 3: lấy 4 với `i = 1` ⇒ trả `5` (khác cha, cách xa).

**Bài 3.** Mức 1: giữ `prev` trong mỗi tầng, reset đầu tầng. Mức 2: gặp `u` trả `prev`. Mức 3: `prev` là `None` nếu `u` đứng đầu tầng.

**Bài 4.** Mức 1: BFS theo tầng thu từng tầng thành list. Mức 2: với tầng `lv`, `res[lv[i].val] = lv[i+1].val`. Mức 3: phần tử cuối tầng ánh xạ tới `None`.

**Bài 5.** Mức 1: như Cách 2, khi gặp `u` còn `i` nút trong tầng. Mức 2: nếu `k ≤ i` thì trả `q[k-1]`. Mức 3: ngược lại `None`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
root = build_tree([1, 2, 3, 4, 5, 6, 7])
b1 = [val_or_none(Solution().findNearestRightNode(root, find_node(root, v))) for v in [5, 3, 7, 1]]
print("Bài 1:", b1)
assert b1 == [6, None, None, None]

# Bài 2
root = build_tree([1, 2, 3, 4, None, None, 5])
assert val_or_none(trace_bfs(root, find_node(root, 4))) == 5

Bài 1: [6, None, None, None]
-- tầng 0: q = [1]
   lấy 1 (i=0), q = [2, 3]
-- tầng 1: q = [2, 3]
   lấy 2 (i=1), q = [3, 4]
   lấy 3 (i=0), q = [4, 5]
-- tầng 2: q = [4, 5]
   lấy 4 (i=1) = u -> còn 1 nút cùng tầng -> trả TreeNode(5)


In [25]:
# Bài 3
def nearest_left_sol(root, u):
    q = deque([root])
    while q:
        prev = None
        for _ in range(len(q)):
            x = q.popleft()
            if x is u:
                return prev
            prev = x
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
    return None

def nearest_left_brute(root, u):
    for lv in levels_nodes(root):
        for i, x in enumerate(lv):
            if x is u:
                return lv[i - 1] if i > 0 else None

def levels_nodes(root):
    res, q = [], deque([root])
    while q:
        cur = []
        for _ in range(len(q)):
            x = q.popleft(); cur.append(x)
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
        res.append(cur)
    return res

rnd = random.Random(3)
for _ in range(200):
    root, nodes = random_tree(rnd.randint(1, 25), rnd)
    for u in nodes:
        assert nearest_left_sol(root, u) is nearest_left_brute(root, u)
print("Bài 3: OK")

Bài 3: OK


In [26]:
# Bài 4
def right_neighbors_sol(root):
    res = {}
    for lv in levels_nodes(root):
        for i, x in enumerate(lv):
            res[x.val] = lv[i + 1].val if i + 1 < len(lv) else None
    return res

print("Bài 4 ví dụ 1:", right_neighbors_sol(build_tree([1, 2, 3, None, 4, 5, 6])))
rnd = random.Random(4)
for _ in range(200):
    root, nodes = random_tree(rnd.randint(1, 25), rnd)
    rn = right_neighbors_sol(root)
    for u in nodes:
        assert rn[u.val] == val_or_none(Solution().findNearestRightNode(root, u))
print("Bài 4: OK")

Bài 4 ví dụ 1: {1: None, 2: 3, 3: None, 4: 5, 5: 6, 6: None}
Bài 4: OK


In [27]:
# Bài 5
def kth_right_sol(root, u, k):
    q = deque([root])
    while q:
        for i in range(len(q) - 1, -1, -1):
            x = q.popleft()
            if x is u:
                return q[k - 1] if k <= i else None
            if x.left: q.append(x.left)
            if x.right: q.append(x.right)
    return None

rnd = random.Random(5)
for _ in range(200):
    root, nodes = random_tree(rnd.randint(1, 25), rnd)
    lvs = levels_nodes(root)
    for u in nodes:
        lv = next(l for l in lvs if any(x is u for x in l))
        pos = next(i for i, x in enumerate(lv) if x is u)
        for k in range(1, 5):
            want = lv[pos + k] if pos + k < len(lv) else None
            assert kth_right_sol(root, u, k) is want
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS theo tầng có "số nút còn lại":**

```python
q = deque([root])
while q:
    for i in range(len(q) - 1, -1, -1):   # i = còn bao nhiêu nút của tầng sau nút này
        x = q.popleft()
        if x is u:
            return q[0] if i else None
        if x.left: q.append(x.left)
        if x.right: q.append(x.right)
return None
```

**Checklist:**
- [ ] So sánh `x is u` (đối tượng), trả về **nút** chứ không phải giá trị?
- [ ] Biết ranh giới tầng (không trả nút của tầng sau)?
- [ ] Đẩy con **trái trước, phải sau** vào hàng đợi (BFS); với DFS stack thì đẩy phải trước?
- [ ] Dùng `deque`, không `list.pop(0)`; không đệ quy sâu?
- [ ] Đã test: cây 1 nút, `u` là gốc, `u` phải nhất có tầng sau không rỗng, hàng xóm khác cha?
- [ ] Đối chiếu chữ ký với đề Premium (doocs: `findNearestRightNode(self, root, u)`).